In [ ]:
from elasticsearch import Elasticsearch

# Connect to Elasticsearch (running in Docker)
es = Elasticsearch("http://localhost:9200", request_timeout=60)

# Define index settings with custom analyzer
# index_settings = {
#     "settings": {
#         "analysis": {
#             "char_filter": {
#                 "remove_spaces": {
#                     "type": "pattern_replace",
#                     "pattern": "\\s+",
#                     "replacement": ""
#                 },
#                 "ocr_fix": {
#                     "type": "mapping",
#                     "mappings": [
#                         "0 => o",
#                         "1 => l",
#                         "5 => s",
#                         "! => i"  # Add more OCR-specific mappings as needed
#                     ]
#                 }
#             },
#             "analyzer": {
#                 "ngram_analyzer": {
#                     "type": "custom",
#                     "char_filter": ["remove_spaces", "ocr_fix"],
#                     "tokenizer": "ngram",
#                     "filter": ["lowercase"],
#                     "min_gram": 2,  # Smaller n-grams for more disruption tolerance
#                     "max_gram": 5   # Larger n-grams for better context
#                 }
#             }
#         }
#     },
#     "mappings": {
#         "properties": {
#             "text": {
#                 "type": "text",
#                 "analyzer": "ngram_analyzer",
#                 "search_analyzer": "ngram_analyzer"
#             }
#         }
#     }
# }


# Create the index (deletes if exists, for testing)
if es.indices.exists(index="asr_index_test"):
    es.indices.delete(index="asr_index_test")
es.indices.create(index="asr_index_test", body=index_settings)

print("Index created successfully!")

ModuleNotFoundError: No module named 'elasticsearch'

In [1]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0,
        "index": {
            "max_ngram_diff": 50   # allow up to 50 difference
        },
        "analysis": {
            "char_filter": {
                "remove_spaces": {
                    "type": "pattern_replace",
                    "pattern": "\\s+",
                    "replacement": ""
                },
                "ocr_fix": {
                    "type": "mapping",
                    "mappings": [
                        "0 => o",
                        "1 => l",
                        "5 => s",
                        "! => i"
                    ]
                }
            },
            "tokenizer": {
                "ngram_tokenizer": {
                    "type": "ngram",
                    "min_gram": 2,
                    "max_gram": 5,
                    "token_chars": ["letter", "digit"]
                }
            },
            "analyzer": {
                "ngram_analyzer": {
                    "type": "custom",
                    "char_filter": ["remove_spaces", "ocr_fix"],
                    "tokenizer": "ngram_tokenizer",
                    "filter": ["lowercase"]
                }
            }
        }
    },
    "mappings": {
        "properties": {
            "text": {
                "type": "text",
                "analyzer": "ngram_analyzer",
                "search_analyzer": "ngram_analyzer"
            }
        }
    }
}


In [7]:
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
import json  # If loading from file

es = Elasticsearch("http://localhost:9200", request_timeout=1000)

# Example docs (replace with your 1M docs, e.g., load from JSON/CSV)
docs = [
    {"_index": "asr_index_test", "_id": "1", "_source": {"text": "ILOVEYO U"}},
    {"_index": "asr_index_test", "_id": "2", "_source": {"text": "YOU LOVE I"}},
    {"_index": "asr_index_test", "_id": "3", "_source": {"text": "END"}},
    {"_index": "asr_index_test", "_id": "4", "_source": {"text": "ILOVEYO-U"}},
    {"_index": "asr_index_test", "_id": "5", "_source": {"text": "ILOVE YOU"}},
]

# For large sets: Load from file (e.g., JSON lines)
# with open('docs.jsonl', 'r') as f:
#     docs = [{"_index": "my_index", "_id": str(i), "_source": json.loads(line)} for i, line in enumerate(f, 1)]

# Bulk index
try:
    success, failed = bulk(es, docs, stats_only=False)
    print(f"Indexed {success} documents")
except Exception as e:
    print("Bulk indexing error:")
    import traceback
    traceback.print_exc()

Indexed 5 documents


In [15]:
from elasticsearch import Elasticsearch

# Initialize the client with increased timeout
es = Elasticsearch(
    "http://localhost:9200",
    request_timeout=30,  # Increase timeout to 30 seconds
    retry_on_timeout=True
)

# Test connection
try:
    if es.ping():
        print("Connected to Elasticsearch!")
        print(es.info())
    else:
        print("Failed to connect to Elasticsearch.")
except Exception as e:
    print(f"Connection error: {e}")

Connected to Elasticsearch!
{'name': '3e65f7bca222', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'cnv2tnvLRou5-lWMnc7bvw', 'version': {'number': '9.0.5', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'abacf6d19441c06668da7264241312caee03cef5', 'build_date': '2025-08-06T22:11:00.741626477Z', 'build_snapshot': False, 'lucene_version': '10.1.0', 'minimum_wire_compatibility_version': '8.18.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


In [ ]:
!ping 127.0.0.1:9200

ping: usage error: Destination address required


ping: 127.0.0.1:9200: Name or service not known


In [27]:
# es.snapshot.create_repository(
#     name="ocr_index",
#     type="fs",
#     settings={
#         "location": "/usr/share/elasticsearch/snapshots",
#         "compress": True
#     }
# )

In [ ]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")

# Define the repository settings
repo_settings = {
    "type": "fs",
    "settings": {
        "location": "/usr/share/elasticsearch/snapshots",
        "compress": True  # Compress snapshot for smaller size
    }
}

# Register the repository
es.snapshot.create_repository(repository="my_backup_repo", body=repo_settings)

print("Snapshot repository created successfully!")

TypeError: Can't use 'repository' and 'body' parameters together because 'repository' is an alias for 'body'. Instead you should only use the 'repository' parameter. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information

In [29]:
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
import json  # If loading from file

es = Elasticsearch("http://localhost:9200")

# Example docs (replace with your 1M docs, e.g., load from JSON/CSV)
docs = [
    {"_index": "ocr_index", "_id": "1", "_source": {"text": "ILOVEYO U"}},
    {"_index": "ocr_index", "_id": "2", "_source": {"text": "YOU LOVE I"}},
    {"_index": "ocr_index", "_id": "3", "_source": {"text": "END"}},
    {"_index": "ocr_index", "_id": "4", "_source": {"text": "ILOVEYO-U"}},
    {"_index": "ocr_index", "_id": "5", "_source": {"text": "ILOVE YOU"}},
]

# For large sets: Load from file (e.g., JSON lines)
# with open('docs.jsonl', 'r') as f:
#     docs = [{"_index": "my_index", "_id": str(i), "_source": json.loads(line)} for i, line in enumerate(f, 1)]

# Bulk index
success, failed = bulk(es, docs)
print(f"Indexed {success} documents, {len(failed)} failed.")

# Refresh index for immediate querying
es.indices.refresh(index="ocr_index")

Indexed 5 documents, 0 failed.


ObjectApiResponse({'_shards': {'total': 2, 'successful': 1, 'failed': 0}})

In [30]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")

# Define query
query_body = {
    "query": {
        "match": {
            "text": "i love you"
        }
    },
    "size": 10  # Number of results
}

# Execute search
response = es.search(index="ocr_index", body=query_body)

# Print ranked results
for hit in response['hits']['hits']:
    print(f"Score: {hit['_score']:.4f}, ID: {hit['_id']}, Text: {hit['_source']['text']}")

Score: 4.4475, ID: 1, Text: ILOVEYO U
Score: 4.4475, ID: 5, Text: ILOVE YOU
Score: 3.7398, ID: 4, Text: ILOVEYO-U
Score: 3.4443, ID: 2, Text: YOU LOVE I
Score: 0.0626, ID: 3, Text: END


In [60]:
!pip install tqdm

  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)


In [48]:
import json
path = "/root/data/frame_asr.json"

# load path
with open(path, 'r') as f:
	data = json.load(f)

len(data)


412909

In [62]:
docs = []
from tqdm import tqdm
for frame_asr in tqdm(data):
	# print(frame_asr)
	idx = frame_asr["idx"]
	asr = frame_asr["asr"]
	docs.append({
		"_index": "asr_index",
		"_id": str(idx),
		"_source": {
			"text": asr
		},
	})

success, failed = bulk(es, docs)
print(f"Indexed {success} documents, {len(failed)} failed.")

# Refresh index for immediate querying
es.indices.refresh(index="asr_index")

100%|██████████| 412909/412909 [00:01<00:00, 360577.55it/s]


Indexed 412909 documents, 0 failed.


ObjectApiResponse({'_shards': {'total': 2, 'successful': 1, 'failed': 0}})

In [68]:
data[1969]

{'idx': 1969,
 'asr': 'tinh Lam Dong cho biet da triet pha mot duong day lua dao ban tien gia de chiem doat tai san bat giu 14 doi tuong Trong do co nam doi tuong cam dau theo dieu tra cac doi tuong da tao cac'}

In [2]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")

# Define query
query_body = {
    "query": {
        "match": {
            "text": "bo doi chu luc thong qua nhieu tran danh"
        }
    },
    "size": 10  # Number of results
}

# Execute search
response = es.search(index="asr_index", body=query_body)

# Print ranked results
for hit in response['hits']['hits']:
    print(f"Score: {hit['_score']:.4f}, ID: {hit['_id']}, Text: {hit['_source']['text']}")

ConnectionError: Connection error caused by: ConnectionError(Connection error caused by: ProtocolError(('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))))

In [70]:
video_metadata_path ="/root/data/videos_metadata.json"
import json
with open(video_metadata_path, 'r') as f:
	video_metadata = json.load(f)

In [77]:
video_metadata[199]

{'author': 'Báo Thanh Niên',
 'channel_id': 'UCIW9cGgoRuGJnky3K3tbzNg',
 'channel_url': 'https://www.youtube.com/channel/UCIW9cGgoRuGJnky3K3tbzNg',
 'description': '#thoisuthanhnien #tinnongthanhnien #phongsuthanhnien \n\nChương trình hướng dẫn học sinh làm bài thi môn Lịch sử kỳ thi tốt nghiệp THPT 2024. Buổi hướng dẫn do thầy Nguyễn Viết Đăng Du, Trường THPT Lê Quý Đôn, TP.HCM thực hiện. \nChương trình được phát trên các kênh thanhnien.vn, facebook.com/thanhnien và YouTube Báo Thanh Niên.\n#tuvanmuathi #tiepsucmuathi #Tuyensinh2024 #chonnganhhocchotuonglai #onthi2024 #biquyetonthi2024 #thithptondautrungdo #baothanhnien\n\n\nNhững thông tin thời sự quốc tế nóng hổi nhất sẽ được Báo Thanh Niên cập nhật trên kênh ĐIỂM NÓNG TOÀN CẦU: youtube.com/@DiemnongtoancauTV\nQuý vị hãy bấm đăng ký kênh ĐIỂM NÓNG TOÀN CẦU để không bỏ lỡ dòng thời sự quốc tế.\n--------\nĐăng kí theo dõi kênh để xem những tin tức mới nhất: \n\nhttp://popsww.com/BaoThanhNien\n\nTin tức báo Thanh Niên - Đọc tin mới onl

In [30]:
from elasticsearch import Elasticsearch

# Connect to Elasticsearch running in Docker (localhost:9200)
es = Elasticsearch(["http://localhost:9200"])

# Check connection
if es.ping():
    print("Connected to Elasticsearch!")
else:
    print("Could not connect.")

# Index a sample document
doc = {
    "title": "Hello Elasticsearch",
    "content": "This is a test document stored in Dockerized Elasticsearch"
}

response = es.index(index="test-index", document=doc)
print("Indexed document:", response['_id'])

# Search for the document
search_body = {
    "query": {
        "match": {
            "content": "test"
        }
    }
}

result = es.search(index="test-index", body=search_body)
print("Search results:", result['hits']['hits'])


Connected to Elasticsearch!


ConnectionTimeout: Connection timed out

In [32]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")

es.info()


ObjectApiResponse({'name': '6b3cf624eaaa', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'RnEJkLaJQhOdWZ_NsmP52w', 'version': {'number': '9.0.5', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'abacf6d19441c06668da7264241312caee03cef5', 'build_date': '2025-08-06T22:11:00.741626477Z', 'build_snapshot': False, 'lucene_version': '10.1.0', 'minimum_wire_compatibility_version': '8.18.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'})